# Этап 3: Обучение Random Forest

Обучаем базовую модель Random Forest на инженерных признаках

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, auc

from train_rf import RandomForestTrainer

sns.set_style('whitegrid')
%matplotlib inline

## 3.1 Загрузка данных

In [ ]:
df = pd.read_csv('../data/features/features.csv')

print(f"Загружено: {df.shape}")
print(f"\nРаспределение классов:")
print(df['label'].value_counts())

## 3.2 Подготовка данных

In [ ]:
trainer = RandomForestTrainer()
X, y = trainer.prepare_features(df)

print(f"\nПризнаков: {X.shape[1]}")
print(f"Примеров: {X.shape[0]}")

In [ ]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

## 3.3 Обучение модели

In [ ]:
# Обучение с GridSearchCV
trainer.train(X_train, y_train, use_grid_search=True)

## 3.4 Оценка модели

In [ ]:
metrics = trainer.evaluate(X_test, y_test)

In [ ]:
# Confusion Matrix
cm = metrics['confusion_matrix']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('../reports/figures/rf_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve
y_pred_proba = trainer.model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig('../reports/figures/rf_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.5 Важность признаков

In [ ]:
importance_df = trainer.get_feature_importance(top_n=20)

print("Топ-20 важных признаков:")
print(importance_df)

In [ ]:
# Визуализация важности
plt.figure(figsize=(10, 8))
plt.barh(range(len(importance_df)), importance_df['importance'], color='steelblue')
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Feature Importance')
plt.title('Топ-20 важных признаков (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.6 Сохранение модели

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

trainer.save_model('../models/rf_model.pkl')
print("✓ Модель сохранена")

## 3.7 Выводы

**Результаты Random Forest:**
- ROC-AUC: [будет заполнено после обучения]
- Precision: [будет заполнено]
- Recall: [будет заполнено]
- F1-Score: [будет заполнено]

**Самые важные признаки:**
1. text_length
2. fake_keywords_count
3. rating
4. unique_words_ratio
5. exclamation_ratio